In [ ]:
from typing import Dict, List

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel  # Requires CUDA

In [ ]:
max_seq_length = 2048  # It can be any value because they support RoPE Scalling
dtype = None
load_in_4bit = True  # Use 4bit quantization to reduce memory usage
model_name = "unsloth/llama-3-8b-bnb-4bit"  # "unsloth/llama-3-8b-Instruct-bnb-4bit",
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

Adding LoRA adapters so we only need to update 1 - 10% of all paramenters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # The dimension of low-rank matrices | Choose any number > 0 | Suggested 8, 16, 32, 64, 128
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,  # The scaling factor for the low-rank matrices
    lora_dropout=0,  # The dropout probability of the LoRA layers | Supports any, but = 0 is optimized
    bias="none",  # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,  # support rank stabilized LoRA
    loftq_config=None,  # And LoftQ
)

In [ ]:
robot_instruct_prompt = """
### Instructions:
Transform input into function calls for controlling industrial robots.

### Input:
{}

### Response:
{}
"""
# Must remember to add the EOS_TOKEN to the tokenized output!!
# Otherwise you'll get infinite generations!
EOS_TOKEN = tokenizer.eos_token


def formatting_prompts_func(examples) -> Dict[str, str]:
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for input, output in zip(inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = robot_instruct_prompt.format(input, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

In [ ]:
repo_id = "Studeni/robot-instructions"
dataset = load_dataset(repo_id, split="train")

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Must remember to add the EOS_TOKEN to the tokenized output!!
# Otherwise you'll get infinite generations!
EOS_TOKEN = tokenizer.eos_token


def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return {
        "text": texts,
    }


pass

from datasets import load_dataset

dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)